<a href="https://colab.research.google.com/github/duartesuzanomarques-coder/Ipython/blob/main/assignment3/kiva_pandas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/kiva_loans.csv')
df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,id,funded_amount,loan_amount,activity,sector,use,country_code,country,region,currency,partner_id,posted_time,disbursed_time,funded_time,term_in_months,lender_count,tags,borrower_genders,repayment_interval,date
0,653051,300.0,300.0,Fruits & Vegetables,Food,"To buy seasonal, fresh fruits to sell.",PK,Pakistan,Lahore,PKR,247.0,2014-01-01 06:12:39+00:00,2013-12-17 08:00:00+00:00,2014-01-02 10:06:32+00:00,12.0,12,NaN,female,irregular,2014-01-01
1,653053,575.0,575.0,Rickshaw,Transportation,to repair and maintain the auto rickshaw used ...,PK,Pakistan,Lahore,PKR,247.0,2014-01-01 06:51:08+00:00,2013-12-17 08:00:00+00:00,2014-01-02 09:17:23+00:00,11.0,14,NaN,"female, female",irregular,2014-01-01
2,653068,150.0,150.0,Transportation,Transportation,To repair their old cycle-van and buy another ...,IN,India,Maynaguri,INR,334.0,2014-01-01 09:58:07+00:00,2013-12-17 08:00:00+00:00,2014-01-01 16:01:36+00:00,43.0,6,"user_favorite, user_favorite",female,bullet,2014-01-01
3,653063,200.0,200.0,Embroidery,Arts,to purchase an embroidery machine and a variet...,PK,Pakistan,Lahore,PKR,247.0,2014-01-01 08:03:11+00:00,2013-12-24 08:00:00+00:00,2014-01-01 13:00:00+00:00,11.0,8,NaN,female,irregular,2014-01-01
4,653084,400.0,400.0,Milk Sales,Food,to purchase one buffalo.,PK,Pakistan,Abdul Hakeem,PKR,245.0,2014-01-01 11:53:19+00:00,2013-12-17 08:00:00+00:00,2014-01-01 19:18:51+00:00,14.0,16,NaN,female,monthly,2014-01-01


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Which countries have the most fertilizer loans above $500?

In [5]:
fert = df[df["use"].str.contains("fertilizer", case=False, na=False) & (df["loan_amount"] > 500)]
print("Number of loans:", len(fert))
fert["country"].value_counts().head(5)

Number of loans: 28642


,count
country,
Kenya,7361
Cambodia,4567
El Salvador,2180
Philippines,1840
Peru,1637


Kenya leads clearly, with almost as many fertilizer loans as Cambodia and El Salvador combined, suggesting that buying inputs is a major need for Kenyan smallholder farmers.

# Which livestock activities get the largest loans?

In [9]:
animals = df[df["activity"].isin(["Pigs", "Cattle", "Dairy", "Poultry"])]
animals.groupby("activity")["loan_amount"].agg(["count", "median"]).sort_values("median", ascending=False)

,count,median
activity,,
Cattle,7865,825.0
Poultry,9783,500.0
Dairy,7914,450.0
Pigs,26624,300.0


Pigs are by far the most common livestock loan, but also the smallest (median $300), while cattle loans are almost three times larger, which makes sense since a cow costs much more than a piglet.

# Which agricultural loans have the longest repayment terms?

In [15]:
agri = df[df["sector"] == "Agriculture"]
agri.sort_values(["term_in_months", "loan_amount"], ascending=[False, False]) \
    [["country", "activity", "loan_amount", "term_in_months"]].head(10)


,country,activity,loan_amount,term_in_months
126839,Mexico,Agriculture,50000.0,144.0
125728,Mexico,Agriculture,27000.0,144.0
121806,Mexico,Agriculture,26750.0,144.0
161636,Kenya,Agriculture,2150.0,122.0
156961,Kenya,Agriculture,1950.0,122.0
161722,Kenya,Agriculture,1950.0,122.0
161700,Kenya,Agriculture,1750.0,122.0
156256,Kenya,Agriculture,1575.0,122.0
161327,Kenya,Farming,1375.0,122.0
162178,Kenya,Agriculture,1375.0,122.0


In [16]:
agri["term_in_months"].median()

14.0

Very long terms are rare outliers: the typical agricultural loan is repaid in about 14 months, but a few loans in Mexico and Kenya stretch to 10–12 years.

# Which countries received the most money in total?

In [17]:
df.groupby("country").agg(total=("loan_amount", "sum"),
                          n_loans=("id", "count"),
                          avg=("loan_amount", "mean")).sort_values("total", ascending=False).head(10)

,total,n_loans,avg
country,,,
Philippines,55342225.0,160441,344.938170
Kenya,34534300.0,75825,455.447412
United States,31496375.0,6093,5169.272116
Peru,30867400.0,22233,1388.359646
Paraguay,29629075.0,11903,2489.210703
El Salvador,26603650.0,39875,667.176176
Cambodia,19872700.0,34836,570.464462
Bolivia,19843250.0,8806,2253.378378
Rwanda,16646300.0,6735,2471.610987


The Philippines receives the most money through a huge number of small loans, while the United States reaches third place with very few but much larger loans (about 15 times the Philippine average).

# Do women-only, men-only and mixed groups borrow different amounts?

In [20]:
df["group_type"] = df["borrower_genders"].apply(
    lambda g: "unknown" if not isinstance(g, str)
    else "female only" if "male" not in g.replace("female", "")
    else "male only" if "female" not in g
    else "mixed")
df.groupby("group_type")["loan_amount"].agg(["count", "mean", "median"])

,count,mean,median
group_type,,,
female only,488081,730.487153,425.0
male only,138523,916.259394,625.0
mixed,40380,1907.064141,1100.0
unknown,4221,1173.655532,625.0


Almost three quarters of Kiva loans go only to women, but their loans are the smallest; mixed groups borrow the most, likely because more people share the loan.

# Which sectors get funded fastest?

In [19]:
df["posted"] = pd.to_datetime(df["posted_time"])
df["funded"] = pd.to_datetime(df["funded_time"])
df["days_to_fund"] = (df["funded"] - df["posted"]).apply(lambda d: d.days)
df.groupby("sector")["days_to_fund"].median().sort_values()

,days_to_fund
sector,
Arts,6.0
Education,6.0
Personal Use,6.0
Manufacturing,6.0
Health,8.0
Retail,9.0
Food,9.0
Entertainment,9.0
Agriculture,10.0


Education and arts loans are funded twice as fast as clothing or transport loans, which suggests lenders prefer projects that feel personal or socially meaningful.

pd.to_datetime (converts text into dates) and .days (number of days in a time difference) were not covered in class.